# Implementation of Chatbot using NLP

In [1]:
pip install nltk scikit-learn streamlit

Note: you may need to restart the kernel to use updated packages.


In [2]:
#Importing necessary libraries
import nltk
import os
import ssl
import streamlit as st
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [3]:
ssl._create_default_https_context = ssl._create_unverified_context
nltk.data.path.append(os.path.abspath('nltk_data'))
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\djnay\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [4]:
intents = [
    {
        "tag": "greeting",
        "patterns": ["Hi", "Hello", "Hey", "How are you", "What's up"],
        "responses": ["Hi there", "Hello", "Hey", "I'm fine, thank you", "Nothing much"]
    },
    {
        "tag": "goodbye",
        "patterns": ["Bye", "See you later", "Goodbye", "Take care"],
        "responses": ["Goodbye", "See you later", "Take care"]
    },
    {
        "tag": "thanks",
        "patterns": ["Thank you", "Thanks", "Thanks a lot", "I appreciate it"],
        "responses": ["You're welcome", "No problem", "Glad I could help"]
    },
    {
        "tag": "about",
        "patterns": ["What can you do", "Who are you", "What are you", "What is your purpose"],
        "responses": ["I am a chatbot", "My purpose is to assist you", "I can answer questions and provide assistance"]
    },
    {
        "tag": "help",
        "patterns": ["Help", "I need help", "Can you help me", "What should I do"],
        "responses": ["Sure, what do you need help with?", "I'm here to help. What's the problem?", "How can I assist you?"]
    },
    {
        "tag": "age",
        "patterns": ["How old are you", "What's your age"],
        "responses": ["I don't have an age. I'm a chatbot.", "I was just born in the digital world.", "Age is just a number for me."]
    },
    {
        "tag": "weather",
        "patterns": ["What's the weather like", "How's the weather today"],
        "responses": ["I'm sorry, I cannot provide real-time weather information.", "You can check the weather on a weather app or website."]
    },
    {
        "tag": "budget",
        "patterns": ["How can I make a budget", "What's a good budgeting strategy", "How do I create a budget"],
        "responses": ["To make a budget, start by tracking your income and expenses. Then, allocate your income towards essential expenses like rent, food, and bills. Next, allocate some of your income towards savings and debt repayment. Finally, allocate the remainder of your income towards discretionary expenses like entertainment and hobbies.", "A good budgeting strategy is to use the 50/30/20 rule. This means allocating 50% of your income towards essential expenses, 30% towards discretionary expenses, and 20% towards savings and debt repayment.", "To create a budget, start by setting financial goals for yourself. Then, track your income and expenses for a few months to get a sense of where your money is going. Next, create a budget by allocating your income towards essential expenses, savings and debt repayment, and discretionary expenses."]
    },
    {
        "tag": "credit_score",
        "patterns": ["What is a credit score", "How do I check my credit score", "How can I improve my credit score"],
        "responses": ["A credit score is a number that represents your creditworthiness. It is based on your credit history and is used by lenders to determine whether or not to lend you money. The higher your credit score, the more likely you are to be approved for credit.", "You can check your credit score for free on several websites such as Credit Karma and Credit Sesame."]
    },
    {
        "tag": "joke",
        "patterns": ["Tell me a joke", "Make me laugh", "Do you know any jokes?"],
        "responses": ["Why don’t skeletons fight each other? Because they don’t have the guts!", 
                      "I told my wife she was drawing her eyebrows too high. She looked surprised!", 
                      "Why did the scarecrow win an award? Because he was outstanding in his field!"]
    },
    {
        "tag": "motivation",
        "patterns": ["Give me motivation", "Motivate me", "I need some inspiration"],
        "responses": ["Believe in yourself! You are stronger than you think.", 
                      "Success is not final, failure is not fatal: It is the courage to continue that counts.", 
                      "Don't watch the clock; do what it does. Keep going."]
    },
    {
        "tag": "time",
        "patterns": ["What time is it?", "Can you tell me the time?", "Current time please"],
        "responses": ["I'm not a clock, but you can check the time on your device.", 
                      "I don’t have a watch, but you do!", 
                      "Just look at the clock, it's always ticking."]
    },
    {
        "tag": "date",
        "patterns": ["What’s today’s date?", "Tell me the date", "What day is it?"],
        "responses": ["I'm not sure, but you can check the calendar!", 
                      "Just check your phone or computer, it's always right.", 
                      "Every day is a great day!"]
    },
    {
        "tag": "fun_fact",
        "patterns": ["Tell me a fun fact", "Do you know any cool facts?", "Give me a random fact"],
        "responses": ["Did you know that honey never spoils? Archaeologists have found pots of honey in ancient Egyptian tombs that are over 3,000 years old and still perfectly edible!", 
                      "Octopuses have three hearts and their blood is blue!", 
                      "Bananas are berries, but strawberries aren't!"]
    },
    {
        "tag": "exercise",
        "patterns": ["How can I stay fit?", "Give me some exercise tips", "Best workouts?"],
        "responses": ["Staying active is key! Try a mix of cardio, strength training, and flexibility exercises.", 
                      "A 30-minute daily walk can do wonders for your health!", 
                      "Consistency is key! Choose an activity you enjoy so you’ll stick with it."]
    },
    {
        "tag": "programming",
        "patterns": ["What is Python?", "Tell me about programming", "Best language to learn?"],
        "responses": ["Python is a popular programming language known for its simplicity and readability.", 
                      "Programming is the process of creating and designing software applications.", 
                      "The best language to learn depends on your goals, but Python, JavaScript, and Java are great choices!"]
    },
    {
        "tag": "food",
        "patterns": ["What should I eat?", "Recommend me a meal", "I’m hungry"],
        "responses": ["How about a nice homemade pasta?", 
                      "Maybe try a fresh salad with grilled chicken!", 
                      "A classic burger and fries never disappoint!"]
    },
    {
        "tag": "travel",
        "patterns": ["Where should I travel?", "Best travel destinations?", "Recommend me a place to visit"],
        "responses": ["Paris for romance, Tokyo for tech, and Bali for relaxation!", 
                      "It depends on what you like! Do you prefer beaches, mountains, or cities?", 
                      "Try exploring a new country or even a hidden gem in your own town!"]
    },
    {
        "tag": "sports",
        "patterns": ["What’s the best sport?", "Tell me about sports", "Best sport to play?"],
        "responses": ["Soccer is the most popular sport worldwide!", 
                      "Basketball, football, and cricket are all great choices!", 
                      "The best sport is the one you enjoy the most!"]
    }
]

In [5]:
# Preprocess intents for training
tags=[]
patterns=[]
responses={}

for intent in intents:
    for pattern in intent['patterns']:
        tags.append(intent['tag'])
        patterns.append(pattern)
    responses[intent['tag']] = intent['responses']

# Vectorize text
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(patterns)
y = tags

# Train model
model = LogisticRegression()
model.fit(X, y)

# Chatbot function
def chatbot_response(user_input):
    user_input_vectorized = vectorizer.transform([user_input])
    tag = model.predict(user_input_vectorized)[0]
    return random.choice(responses[tag])

# Streamlit UI
st.write("Type a message and get a response!")
user_input = st.text_input("You:", "")

if user_input:
    bot_response = chatbot_response(user_input)
    st.write(f"Chatbot: {bot_response}")

2025-03-02 10:13:53.331 
  command:

    streamlit run C:\Users\djnay\AppData\Roaming\Python\Python311\site-packages\ipykernel_launcher.py [ARGUMENTS]
2025-03-02 10:13:53.333 Session state does not function when running a script without `streamlit run`
